In [3]:
import pandas as pd
import numpy as np

### 외부 csv파일을 DataFram으로 가지고 오기
파일 경로는 잘 확인해주셔야 합니다

In [4]:
def csv_to_df(file_name, sep_type='|'):
    '''
    외부 txt, csv 파일을 가지고 와서 데이터 프레임화 시키기
    '''
    df = None
    file_path = f'../../공공/Data/{file_name}'

    try:
        # engine='python'을 명시적으로 지정
        # quoting=3 (csv.QUOTE_NONE)을 추가하여 따옴표 무시
        df = pd.read_csv(file_path, sep=sep_type, engine='python', quoting=3)
        print("파일이 성공적으로 불러와졌습니다!")
        print(df.head())		# 파일 5개 미리보기

    except FileNotFoundError:
        print(f"오류: '{file_path}' 경로에 파일이 존재하지 않습니다.")

    except Exception as e:
        print(f"오류가 발생했습니다: {e}")

    return df


### Column 순서를 영단어, 한국어 순서로 바꾸기(한국어, 영어인 경우)

In [5]:
def swap_columns(df):
    """
    데이터프레임의 1열과 2열의 내용이 영어-한글 순서로 잘못된 경우,
    그 순서를 올바르게 바꾸는 함수.
    
    Args:
        df (pd.DataFrame): 1열과 2열로 구성된 데이터프레임.
        
    Returns:
        pd.DataFrame: 순서가 수정된 데이터프레임.
    """
    # 임시로 컬럼 이름을 지정합니다.
    df.columns = ['col1', 'col2']
    
    # 1열이 한글이고 2열이 영어인 행을 찾아 순서를 바꿉니다.
    # 한글 문자는 '가'~'힣' 유니코드 범위에 속합니다.
    df['is_korean_col1'] = df['col1'].astype(str).str.contains(r'[가-힣]')
    df['is_english_col2'] = df['col2'].astype(str).str.contains(r'[a-zA-Z]')
    
    # 순서를 바꿔야 하는 행의 인덱스를 찾습니다.
    # 1열에 한글이 있고 2열에 영어가 있을 때 (일반적으로)
    swap_indices = df[(df['is_korean_col1']) & (df['is_english_col2'])].index
    
    # 해당 행의 컬럼 값들을 교환합니다.
    df.loc[swap_indices, ['col1', 'col2']] = df.loc[swap_indices, ['col2', 'col1']].values
    
    # 임시 컬럼을 제거합니다.
    df = df.drop(columns=['is_korean_col1', 'is_english_col2'])
    
    return df

In [6]:
all2_removed_df = csv_to_df('all2_removed_modified.txt')
all2_removed_df = swap_columns(all2_removed_df)
all2_removed_df.columns = ['영단어', '한국단어']

파일이 성공적으로 불러와졌습니다!
                       영단어                한국단어
0              AC BALANCER              교류 밸런서
1             AC GENERATOR    교류 발전기 ( 交流發電機 )
2                 AC MOTOR              교류 전동기
3                 AC MOTOR  교류 정류자기 ( 交流整流子機 )
4  AC SLNGLE SPEED CONTROL         교류 1 단 속도제어


In [7]:
en_ko_desc_df = csv_to_df('en_ko_desc_modified.txt')
en_ko_only_df = csv_to_df('en_ko_only_modified.txt')
en_ko_pairs_df = csv_to_df('en_ko_pairs_modified.txt')
IMO_df = csv_to_df('IMO_word.csv')

파일이 성공적으로 불러와졌습니다!
                                영단어     한국단어  \
0                     Factory Shipa      가공선   
1               Malleable Cast Iron     가단주철   
2                        Guard Rail     가드레일   
3  Variable Voltage Welding Machine  가변전압용접기   
4                            Canopy      가설지   

                                                  설명  
0        선내에 수산물 가공설비를 갖추고 어선에서 수획한 어류를 선상에서 가공하는 선박  
1  백주철을 열처리하여 내부에 함유된 탄소를 제거하여 가단성(Malleability)을...  
2                          추락이나 위험접근을 방지하는 손잡이 또는 레일  
3                    전류의 증가에 따라 전압이 자동변화하는 형식의 아크용접기  
4                      무갑판선에서 풍우를 막기 위하여 설치하는 범포 가리개  
파일이 성공적으로 불러와졌습니다!
                   영단어       한국단어  설명
0      ABOVE BASE LINE      기선 상부 NaN
1      ABOVE BASE LINE     기준선 상부 NaN
2  ACCEPT WITH COMMENT  주석조건하에 수락 NaN
3     ACCEPTANCE TRIAL     인수 시운전 NaN
4        ACCOMMODATION        거주구 NaN
파일이 성공적으로 불러와졌습니다!
                         영단어         한국단어  설명
0  A/C (Anticorrosive) Paint         방청도료 NaN
1   

### Pandas의 concat함수를 사용하여 데이터 프레임 합치기

In [8]:
df_list = [all2_removed_df, en_ko_desc_df, en_ko_only_df, en_ko_pairs_df, IMO_df]

# 함수 실행
combined_df = pd.concat(df_list, axis=0, ignore_index=True)

In [9]:
combined_df.describe(include='all')

,영단어,한국단어,설명
count,8739,8739,3689
unique,8128,7709,3680
top,FUSE,보강재,해군장교의 계급명
freq,14,4,4


In [10]:
combined_df['영단어'].nunique()   # 고유한 영단어는 8128개

8128

### 중복된 값들 제거하기

In [11]:
# 1. 'col3' 열을 기준으로 NaN 값이 아닌 행이 먼저 오도록 정렬
#    (NaN은 True/False 비교 시 False로 간주되므로, isna()를 사용)
combined_df['설명_temp'] = ~combined_df['설명'].isna()
combined_df_sorted = combined_df.sort_values(by=['영단어', '한국단어', '설명_temp'], ascending=[True, True, False])

# 임시로 생성한 컬럼 제거
combined_df_sorted = combined_df_sorted.drop(columns='설명_temp')

# 2. 'col1'과 'col2'를 기준으로 중복 제거
#    keep='first'가 기본값이므로 정렬된 순서에서 뒤에 있는 중복 행 삭제
# subset = ['영단어'] : 고유한 영단어만 남음(8128)
# subset = ['영단어', '한국단어']: 영단어와 한국단어의 매칭이 고유한 값만 남음(8575)
final1_df = combined_df_sorted.drop_duplicates(subset=['영단어'], keep='first', ignore_index=True)
final2_df = combined_df_sorted.drop_duplicates(subset=['영단어', '한국단어'], keep='first', ignore_index=True)

In [ ]:
# 1번은 고유한 영단어만으로 8128개
# 2번은 영단어와 한국단어의 고유한 조합으로 8575개
final1_df.count(), final2_df.count()

(영단어     8128
 한국단어    8128
 설명      3480
 dtype: int64,
 영단어     8575
 한국단어    8575
 설명      3684
 dtype: int64)

### 합친 데이터프레임을 csv로 다시 추출하기

In [13]:
save_file_path1 = '../../공공/Data/total_data/eng_word.csv'

final1_df.to_csv(save_file_path1, sep='|', index=False, encoding='utf-8', na_rep='NULL')

In [14]:
save_file_path2 = '../../공공/Data/total_data/eng_kor_word.csv'

final2_df.to_csv(save_file_path2, sep='|', index=False, encoding='utf-8', na_rep='NULL')